@file     DQC1.ipynb â€” logging setup cell  
@brief    Dual-output logger: writes every dlog() call to stdout and a  
          timestamped log file under logs/log_<yyyymmddhhmm>.txt.  
@license  MIT License â€” see LICENSE file for details.  

In [ ]:
import os
import datetime

# Create logs/ directory next to the notebook if it does not exist yet
os.makedirs("logs", exist_ok=True)

# Build a timestamped filename so every run produces its own log
_log_ts   = datetime.datetime.now().strftime("%Y%m%d%H%M")
_log_path = os.path.join("logs", f"log_{_log_ts}.txt")

# Open line-buffered so each dlog() call is flushed to disk immediately
_log_file = open(_log_path, "w", buffering=1)


def dlog(msg: str = "") -> None:
    """@brief Write a message to stdout and the session log file simultaneously.

    @param msg  The message string to emit; written as a single line.
    @note  The file is opened line-buffered so every call flushes immediately,
        making it safe to tail -f the log during a long notebook run.
    """
    # Mirror the message to the console so notebook output is unchanged
    print(msg)
    # Write to the log file with a newline; buffering=1 ensures immediate flush
    _log_file.write(msg + "\n")
# end def dlog


# Announce the session start so the log is never empty
dlog(f"[LOG] Session started : {datetime.datetime.now().isoformat()}")
dlog(f"[LOG] Writing log to  : {_log_path}")


In [ ]:


# ============================================================
# DQC1 Entropy Clustering (Î”H quantum kernel)
# ------------------------------------------------------------
#   Feature map circuits â†’ projected-fidelity kernel matrix â†’
#   entropy-driven cluster assignment â†’ optional entropy reduction
# ============================================================

import os
import sys
import time
import math
import random
import contextlib
from collections import Counter
from typing import List, Optional

import numpy as np

from sklearn.decomposition import PCA
from sklearn.metrics import (
    normalized_mutual_info_score,
    fowlkes_mallows_score,
)

try:
    import torch as t
    _TORCH_AVAILABLE = True
except ImportError:
    t = None
    _TORCH_AVAILABLE = False
    print("[WARN] torch not available (ARM64) â€” tensor path disabled")

# Qiskit imports
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector
try:
    from qiskit_aer import Aer, AerSimulator
    _AER_AVAILABLE = True
except ImportError:
    Aer = None
    AerSimulator = None
    _AER_AVAILABLE = False
    print("[WARN] qiskit-aer not available (ARM64) â€” local shot-based simulation disabled")

# ===== CONFIG =====
FEATURE_MAP = os.environ.get("FEATURE_MAP", "zz")  # zz | karimi | zdiag | rxryrz | pauli
LAYERS = int(os.environ.get("LAYERS", "3"))  # Circuit depth; more layers = more expressive, higher transpile cost

INIT_CLUSTERS = int(os.environ.get("INIT_CLUSTERS", "6"))  # Initial number of clusters for DQC1 Î”H seeding
N_INIT = int(os.environ.get("N_INIT", "10"))  # Minimum samples per cluster before Î”H assignment begins

DQC1_KERNEL_SHOTS = int(os.environ.get("DQC1_KERNEL_SHOTS", "1024"))
DQC1_EXACT_KERNEL = int(os.environ.get("DQC1_EXACT_KERNEL", "1"))
DQC1_COMP_RANK = int(os.environ.get("DQC1_COMP_RANK", "0"))  # Disabled â€” do not compress IBM quantum kernel


USE_PCA = bool(int(os.environ.get("USE_PCA", "1")))
PCA_COMPONENTS = int(os.environ.get("PCA_COMPONENTS", "6"))

DATA_NOISE = float(os.environ.get("DATA_NOISE", "0.0"))
RANDOM_STATE = int(os.environ.get("RANDOM_STATE", "42"))

DISABLE_TIMERS = int(os.environ.get("DISABLE_TIMERS", "0"))

USE_IBM_BACKEND = bool(int(os.environ.get("USE_IBM_BACKEND", "0")))  # Set to 1 to route kernel to IBM Quantum hardware
N_SAMPLES = int(os.environ.get("N_SAMPLES", "0"))  # 0 = full dataset; >0 = subsample train set for small IBM test runs
DISABLE_REDUCTION = bool(int(os.environ.get("DISABLE_REDUCTION", "1")))  # 1 = skip entropy reduction; keeps K=INIT_CLUSTERS from init phase


# ===== UTILITIES =====

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if _TORCH_AVAILABLE:
    t.manual_seed(RANDOM_STATE)

if DISABLE_TIMERS:
    @contextlib.contextmanager
    def Timer(msg: str):
        yield
else:
    class Timer:
        def __init__(self, msg: str):
            self.msg = msg
        def __enter__(self):
            print(f"[TIMER] {self.msg}â€¦")
            self.t0 = time.time()
            return self
        def __exit__(self, *args):
            dt = time.time() - self.t0
            print(f"[TIMER] {self.msg} â†’ {dt:.2f}s")
# end if DISABLE_TIMERS

# ============================================================
# FEATURE MAPS & QUANTUM KERNEL
# ============================================================
def get_feature_map_circuit(x: np.ndarray, kind: str, layers: int) -> QuantumCircuit:
    """@brief Build quantum feature map circuit U(x) for DQC1 kernel and clustering.

    @param x      Input feature vector (1D numpy array).
    @param kind   String specifying the type of feature map ('karimi', 'zz', etc).
    @param layers Number of circuit layers (depth).
    @return       QuantumCircuit encoding the feature vector as a unitary.
    @note         Used for constructing the DQC1 kernel and quantum clustering.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    kind = (kind or "karimi").strip().lower()

    if kind == "karimi":
        # Karimi-style feature map: Hadamard + RZ + RZZ entanglement
        qc = QuantumCircuit(n, name="U_karimi")
        qc.h(range(n))  # Place all qubits in superposition
        for i in range(n):
            qc.rz(2 * np.pi * x[i], i)  # Encode features as Z rotations
        if n >= 2:
            for i in range(n):
                for j in range(i + 1, n):
                    qc.rzz(2 * np.pi * x[i] * x[j], i, j)  # Pairwise entanglement
        return qc

    elif kind == "zz":
        # ZZ feature map: multiple layers of RZ and RZZ gates
        qc = QuantumCircuit(n, name="U_zz")
        qc.h(range(n))
        for L in range(layers):
            for q in range(n):
                qc.rz(2*np.pi*x[q], q)  # Encode features at each layer
            for i in range(n):
                for j in range(i+1, n):
                    qc.rzz(2*np.pi*x[i]*x[j], i, j)  # Layered entanglement
        return qc

    elif kind == "zdiag":
        # Z-diagonal feature map: RZ rotations and CZ entanglement in a brickwork pattern
        qc = QuantumCircuit(n, name="U_zdiag")
        for L in range(layers):
            for q in range(n):
                qc.rz(2*np.pi*x[q], q)
            for q in range(0, n-1, 2):
                qc.cz(q, q+1)  # Even pairs
            for q in range(1, n-1, 2):
                qc.cz(q, q+1)  # Odd pairs
        return qc

    elif kind == "rxryrz":
        # RX-RY-RZ feature map: richer single-qubit rotations and CX entanglement
        qc = QuantumCircuit(n, name="U_rxryrz")
        for L in range(layers):
            for q in range(n):
                a = x[q]
                qc.rx(2*np.pi*(a+0.1*L), q)
                qc.ry(2*np.pi*(a*a+0.2*L), q)
                qc.rz(2*np.pi*(0.3*a+0.05*L), q)
            for q in range(n-1):
                qc.cx(q, q+1)  # Chain entanglement
        return qc

    elif kind == "pauli":
        # Pauli feature map: nonlinear RZ and RZZ gates for expressivity
        qc = QuantumCircuit(n, name="U_pauli")
        for L in range(layers):
            for q in range(n):
                qc.rz(2*np.pi*(0.6*x[q] + 0.2*(x[q]**3)), q)
            for i in range(n):
                for j in range(i+1, n):
                    qc.rzz(2*np.pi*(0.4*x[i]*x[j] + 0.1*(x[i]**2 + x[j]**2)), i, j)
        return qc

    else:
        # Default to Karimi feature map if unknown kind
        qc = QuantumCircuit(n, name="U_karimi")
        qc.h(range(n))
        for i in range(n):
            qc.rz(2 * np.pi * x[i], i)
        if n >= 2:
            for i in range(n):
                for j in range(i + 1, n):
                    qc.rzz(2 * np.pi * x[i] * x[j], i, j)
        return qc
# end def get_feature_map_circuit


def dqc1_kernel_value(xi: np.ndarray, xj: np.ndarray, shots: int = 1024, exact: bool = True) -> float:
    """@brief Compute DQC1 projected-fidelity kernel value between two samples.

    @param xi     First input feature vector (1D numpy array).
    @param xj     Second input feature vector (1D numpy array).
    @param shots  Number of measurement shots (used when exact=False and AerSimulator available).
    @param exact  If True, use statevector simulation for exact fidelity.
    @return       Projected-fidelity kernel value in [0, 1].
    @note         Uses <0|U_i U_jâ€ |0> overlap as the kernel value.
    """
    # Build quantum feature map circuits for both samples
    U_i = get_feature_map_circuit(xi, FEATURE_MAP, LAYERS)
    U_j = get_feature_map_circuit(xj, FEATURE_MAP, LAYERS)

    # Compute relative unitary U_i @ U_jâ€  so that <0|U_rel|0> gives the fidelity kernel
    U_rel = U_i.copy()
    U_rel.compose(U_j.inverse(), inplace=True)

    if exact:
        # Use statevector simulation to compute the overlap with |0...0>
        psi = Statevector.from_instruction(U_rel)
        sv_data = psi.data
        # Fidelity kernel: K = |<0|U_rel|0>|^2 = |sv[0]|^2
        v = float(np.abs(sv_data[0]) ** 2)
    else:
        # Shot-based path: run U_rel on AerSimulator and count the all-zeros outcome
        if _AER_AVAILABLE:
            # Add measurements so AerSimulator produces bitstring counts
            qc_meas = U_rel.copy()
            qc_meas.measure_all()
            _sim = AerSimulator(method="statevector")
            _tqc = transpile(qc_meas, backend=_sim)
            _counts = _sim.run(_tqc, shots=shots).result().get_counts()
            zero_key = "0" * U_rel.num_qubits
            v = float(_counts.get(zero_key, 0)) / shots
        else:
            # AerSimulator not available (ARM64); fall back to exact statevector
            psi = Statevector.from_instruction(U_rel)
            sv_data = psi.data
            v = float(np.abs(sv_data[0]) ** 2)

    # Ensure kernel value is in [0, 1]
    return float(np.clip(v, 0.0, 1.0))
# end def dqc1_kernel_value


# ============================================================
# DQC1 ENTROPY CLUSTERING CLASS
# ============================================================

class DQC1EntropyCluster:
    """DQC1-based entropy clustering with init + optional reduction."""

    def __init__(self,
                 init_clusters: int = INIT_CLUSTERS,
                 n_init_per_seed: int = N_INIT,
                 kernel_shots: int = DQC1_KERNEL_SHOTS,
                 exact_sv_kernel: bool = True,
                 random_state: int = RANDOM_STATE,
                 compress_rank: Optional[int] = None):
        self.K0 = int(init_clusters)
        self.N_INIT = int(n_init_per_seed)
        self.kernel_shots = int(kernel_shots)
        self.exact_sv_kernel = bool(exact_sv_kernel)
        self.random_state = int(random_state)
        self.rng = np.random.RandomState(self.random_state)
        self.compress_rank = int(compress_rank) if compress_rank is not None else None
        self.labels_full_ = None
        self.Kstar_ = None

    def _compute_full_kernel(self, X_np):
        """Build DQC1 kernel matrix."""
        # Delegate to the BLAS-vectorized kernel builder
        K = build_dqc1_kernel_matrix(
            X_np,
            shots=self.kernel_shots,
            exact=self.exact_sv_kernel,
        )
        K = 0.5 * (K + K.T)  # Symmetrize
        K = np.clip(K, 0.0, 1.0)
        return K
    # end def _compute_full_kernel

    def _H2_cluster_fast(self, M_full: np.ndarray, idxs: List[int]) -> float:
        """Compute entropy for a cluster using kernel matrix trace."""
        if len(idxs) <= 1:
            return 0.0
        sub = M_full[np.ix_(idxs, idxs)]
        trace_val = float(np.trace(sub))
        if trace_val <= 0:
            return 0.0
        return float(-np.log(trace_val + 1e-12))
    # end def _H2_cluster_fast

    def _S2_total_fast(self, M_full: np.ndarray, clusters) -> float:
        """Sum entropy across all clusters."""
        return sum(self._H2_cluster_fast(M_full, c) for c in clusters if c)
    # end def _S2_total_fast

    def _delta_H_fast(self, M_full: np.ndarray, base_idxs: List[int], i: int) -> float:
        """Change in entropy when adding sample i to cluster."""
        H_now = self._H2_cluster_fast(M_full, base_idxs) if base_idxs else 0.0
        H_new = self._H2_cluster_fast(M_full, base_idxs + [i])
        return H_new - H_now
    # end def _delta_H_fast

    def _min_dist_to_cluster_members(self, X_np: np.ndarray, unassigned, clusters):
        """Find unassigned sample closest to any cluster."""
        best_i, best_c, best_d2 = None, None, float("inf")
        X_u = X_np[np.asarray(unassigned, dtype=int)]

        for c_id, c in enumerate(clusters):
            if not c:
                continue
            X_c = X_np[np.asarray(c, dtype=int)]
            d2_mat = ((X_u[:, None, :] - X_c[None, :, :]) ** 2).sum(axis=2)
            d2_min = d2_mat.min(axis=1)
            j = int(np.argmin(d2_min))
            cand_d2 = float(d2_min[j])
            if cand_d2 < best_d2:
                best_d2 = cand_d2
                best_i = int(np.asarray(unassigned, dtype=int)[j])
                best_c = c_id
        return best_i, best_c
    # end def _min_dist_to_cluster_members

    def fit(self, X_np: np.ndarray):
        """Init + Î”H assignment only (no reduction)."""
        X_np = np.asarray(X_np, float)
        N = X_np.shape[0]

        # Compute kernel
        with Timer(f"DQC1 kernel (fmap={FEATURE_MAP}, L={LAYERS})"):
            self.M_full = self._compute_full_kernel(X_np)

        # Initialize clusters with random seeds
        seeds = self.rng.choice(N, self.K0, replace=False)
        clusters = [[int(s)] for s in seeds]
        unassigned = np.setdiff1d(np.arange(N), seeds, assume_unique=True).tolist()

        # Grow seeds to minimum size
        while any(len(c) < self.N_INIT for c in clusters) and unassigned:
            i_star, c_star = self._min_dist_to_cluster_members(X_np, unassigned, clusters)
            clusters[c_star].append(i_star)
            unassigned.remove(i_star)

        # Assign remaining by Î”H
        while unassigned:
            i_star, _ = self._min_dist_to_cluster_members(X_np, unassigned, clusters)
            best_k, best_dH = None, float("inf")
            for k, c in enumerate(clusters):
                dH = self._delta_H_fast(self.M_full, c, i_star)
                if dH < best_dH:
                    best_k, best_dH = k, dH
            clusters[best_k].append(i_star)
            unassigned.remove(i_star)

        self.clusters_ = clusters
        self.X_ = X_np
        return self
    # end def fit

    def fit_full(self, X_np: np.ndarray):
        """Init + Î”H + optional reduction (with K* selection)."""
        X_np = np.asarray(X_np, float)

        # Do init if not already done
        if not hasattr(self, 'X_') or self.X_.shape[0] != X_np.shape[0]:
            self.fit(X_np)

        # Apply reduction (or skip if disabled)
        clusters_final = self._reduce(self.X_, self.clusters_)

        # Select K* (use len(clusters) if reduction disabled)
        self.Kstar_ = len(clusters_final)

        # Create labels
        N = self.X_.shape[0]
        self.labels_full_ = -np.ones(N, dtype=int)
        for k, c in enumerate(clusters_final):
            for idx in c:
                self.labels_full_[idx] = k

        return self
    # end def fit_full

    def _reduce(self, X_arr: np.ndarray, clusters: List[List[int]]) -> List[List[int]]:
        """Reduction phase (can be overridden to disable)."""
        # Default: use fast reduction or no-op
        clusters = [list(c) for c in clusters if c]
        self.entropy_log_ = [self._S2_total_fast(self.M_full, clusters)]
        return clusters
    # end def _reduce
# end class DQC1EntropyCluster


def _as_controllable_gate(U):
    """Convert U to a controllable gate with definition."""
    from qiskit.circuit import Gate
    from qiskit import QuantumCircuit

    if isinstance(U, QuantumCircuit):
        return U.to_gate(label=getattr(U, "name", "U_A"))

    if hasattr(U, "definition") and U.definition is not None:
        g = Gate(getattr(U, "name", "U_A"), U.num_qubits, [])
        g.definition = U.definition
        return g

    if hasattr(U, "to_gate"):
        g = U.to_gate()
        if getattr(g, "definition", None) is None:
            raise TypeError("Block-encoding has no definition; cannot simulate.")
        return g

    raise TypeError(f"Cannot convert {type(U)} to a controllable gate with definition.")
# end def _as_controllable_gate




In [ ]:
# Cell: Optimized DQC1 kernel builder (vectorized BLAS)
# Fully removes per-pair scheduling/printing; computes all pairs via matrix multiply.

# Reuse existing utilities and feature map imports from earlier cells:
# - get_feature_map_circuit
# - Statevector
# - dqc1_kernel_value
# - Timer
# - FEATURE_MAP, LAYERS


def build_dqc1_kernel_matrix(
    X_np: np.ndarray,
    shots: int = 1024,
    exact: bool = True,
    n_workers: int = 1,
    gpu_ids: Optional[List[int]] = None,
) -> np.ndarray:
    """@brief Build the DQC1 projected-fidelity kernel matrix.

    @param X_np     Input feature matrix (N, d) as numpy array.
    @param shots    Number of measurement shots (unused if exact=True).
    @param exact    If True, use statevector simulation for exact fidelity.
    @param n_workers Number of workers for parallel computation (unused in this version).
    @param gpu_ids  List of GPU IDs (unused in this version).
    @return         Symmetric kernel matrix (N, N) with values in [0, 1].
    @note           Vectorized exact path uses BLAS for speed. Shot-based path is serial.
    """
    # Ensure input is a float numpy array for consistent processing
    X_np = np.asarray(X_np, float)
    N = len(X_np)

    if exact:
        # Precompute statevectors for all samples to enable vectorized operations
        with Timer("DQC1 kernel: precompute statevectors"):
            sv_list = [Statevector.from_instruction(get_feature_map_circuit(X_np[i], FEATURE_MAP, LAYERS)).data for i in range(N)]
            # Stack into a complex matrix for efficient BLAS operations
            S = np.vstack([np.asarray(sv, dtype=np.complex128) for sv in sv_list])
        # Compute the complex Gram matrix using matrix multiplication for all pairwise overlaps
        with Timer("DQC1 kernel (vectorized BLAS)"):
            G = S @ S.conj().T  # Inner product matrix of statevectors
            K = np.abs(G) ** 2  # Fidelity kernel as squared absolute value of overlaps
            # Force diagonal to 1.0 for numerical stability and clip to valid range
            np.fill_diagonal(K, 1.0)
            K = np.clip(K.real, 0.0, 1.0)
        return K

    # Shot-based path: initialize symmetric kernel matrix with self-fidelity
    K = np.zeros((N, N), dtype=float)
    for i in range(N):
        K[i, i] = 1.0
    # Generate upper triangular pairs to avoid redundant computations
    pairs = [(i, j) for i in range(N) for j in range(i + 1, N)]

    def _one_shots(i: int, j: int) -> float:
        # Compute kernel value using shot-based simulation for a single pair
        v = dqc1_kernel_value(X_np[i], X_np[j], shots=shots, exact=False)
        return float(np.clip(v, 0.0, 1.0))

    # Serially compute kernel for each pair and mirror to lower triangle
    with Timer("DQC1 kernel (serial, shots)"):
        for i, j in pairs:
            v = _one_shots(i, j)
            K[i, j] = K[j, i] = v
    return K
# end def build_dqc1_kernel_matrix

In [ ]:
# ============================================================
# IBM CLOUD SECTION
# ============================================================
# This cell authenticates with IBM Quantum and overrides
# build_dqc1_kernel_matrix() to route circuit execution to IBM
# hardware when USE_IBM_BACKEND=1. When USE_IBM_BACKEND=0 (default)
# the local Statevector path defined in Cell 3 remains active.
# ============================================================

import os

# USE_IBM_BACKEND is set in Cell 2 config block (env-driven); read here

if USE_IBM_BACKEND:
    from qiskit_ibm_runtime import QiskitRuntimeService
    from ibm_kernel import build_ibm_kernel_matrix
    from ibm_backend import select_backend

    # Authenticate using saved credentials (saved via QiskitRuntimeService.save_account)
    _ibm_service = QiskitRuntimeService(channel="ibm_cloud")

    # Select best available backend from preferred list (ibm_fez â†’ ibm_torino â†’ ...)
    _ibm_backend = select_backend(_ibm_service)
    print(f"[IBM] Connected. Using backend: {_ibm_backend.name} "
          f"({_ibm_backend.num_qubits} qubits)")

    # Override build_dqc1_kernel_matrix() so DQC1EntropyCluster._compute_full_kernel()
    # transparently routes to IBM hardware without any other code changes
    def build_dqc1_kernel_matrix(
        X_np,
        shots=DQC1_KERNEL_SHOTS,
        exact=True,
        n_workers=1,
        gpu_ids=None,
    ):
        """@brief IBM-backed kernel matrix builder (overrides local Statevector version).

        Drop-in replacement active when USE_IBM_BACKEND=1. Delegates to
        build_ibm_kernel_matrix() in ibm_kernel.py, passing the notebook's
        feature map function and config globals so ibm_kernel.py stays
        decoupled from notebook state.

        @param X_np     (N, d) PCA-reduced, scaled feature matrix.
        @param shots    Measurement shots per circuit (from DQC1_KERNEL_SHOTS).
        @param exact    Ignored on IBM path (always shot-based on real hardware).
        @param n_workers Ignored on IBM path (batching handled by ibm_kernel.py).
        @param gpu_ids  Ignored on IBM path.
        @return         (N, N) symmetric kernel matrix with diagonal = 1.0.
        """
        # Pass the pre-selected backend and notebook globals to ibm_kernel.py;
        # backend is pre-selected above so no second select_backend() call occurs
        return build_ibm_kernel_matrix(
            X_np,
            feature_map_fn=get_feature_map_circuit,
            feature_map=FEATURE_MAP,
            layers=LAYERS,
            shots=shots,
            backend=_ibm_backend,  # use pre-selected backend; avoids redundant re-selection
        )
    # end def build_dqc1_kernel_matrix

    print("[IBM] Kernel routing ENABLED â€” build_dqc1_kernel_matrix() â†’ IBM SamplerV2")

else:
    # Local path: build_dqc1_kernel_matrix() defined in Cell 3 remains active
    print("[IBM] Kernel routing DISABLED â€” using local Statevector (USE_IBM_BACKEND=0)")
# end if USE_IBM_BACKEND

In [ ]:
# ============================================================
# QPU COMPUTE TIME ESTIMATION â€” FakeTorino
# ============================================================
# Builds the actual kernel circuit (U_i @ U_jâ€ ) used in deployment,
# transpiles it with ASAP scheduling to obtain true duration in dt
# units, then extrapolates to the full kernel matrix run.
#
# Run BEFORE deploying to IBM hardware to confirm the job fits
# within QPU time limits. Does NOT submit any real jobs.
#
# NOTE on input vector size vs QPU time:
#   Raw input dimension (e.g. 256 or 1024 features) does NOT affect
#   QPU time. PCA always reduces the input to PCA_COMPONENTS dimensions
#   before the circuit is built. The circuit qubit count = PCA_COMPONENTS,
#   so PCA_COMPONENTS is the variable that drives circuit depth and QPU time.
# ============================================================

from qiskit import QuantumCircuit
from qiskit import transpile as qiskit_transpile
from qiskit_ibm_runtime.fake_provider import FakeTorino

# IBM open plan QPU time budget in seconds (10 minutes per month, as of 2026)
IBM_QPU_LIMIT_SECONDS = 600

_fake = FakeTorino()


def _circuit_duration_seconds(n_components: int, feature_map: str,
                               layers: int, fake_backend) -> float:
    """@brief Transpile a kernel circuit and return its duration in seconds per shot.

    @param n_components   Number of PCA components (= qubits).
    @param feature_map    Feature map kind string (e.g. "zz").
    @param layers         Number of feature map layers.
    @param fake_backend   FakeTorino instance for transpilation.
    @return               Seconds per shot, or -1.0 if scheduling is unavailable.
    @note  Builds U_i @ U_jâ€  with zero vectors as stand-in inputs; circuit
           structure (depth, gate count) is independent of feature values.
    """
    # Use zero vectors â€” circuit structure does not depend on feature values
    xi = np.zeros(n_components)
    xj = np.zeros(n_components)
    U_i = get_feature_map_circuit(xi, feature_map, layers)
    U_j = get_feature_map_circuit(xj, feature_map, layers)

    # Build U_i @ U_jâ€  with measurements â€” exactly what ibm_kernel.py submits
    qc = QuantumCircuit(n_components, n_components)
    qc.compose(U_i, inplace=True)
    qc.compose(U_j.inverse(), inplace=True)
    qc.measure(range(n_components), range(n_components))

    try:
        # scheduling_method="asap" annotates .duration in dt ticks
        tqc = qiskit_transpile(qc, backend=fake_backend,
                               optimization_level=1, scheduling_method="asap")
        return tqc.duration * fake_backend.dt
    except Exception:
        # Fallback: count 2-qubit gates Ã— 500 ns each
        tqc = qiskit_transpile(qc, backend=fake_backend, optimization_level=1)
        ops = tqc.count_ops()
        cx = ops.get("cx", 0) + ops.get("ecr", 0)
        return cx * 500e-9
# end def _circuit_duration_seconds


def _qpu_seconds(circuit_duration_s: float, n_samples: int,
                 shots: int = DQC1_KERNEL_SHOTS) -> float:
    """@brief Total QPU seconds for a full kernel matrix run.

    @param circuit_duration_s  Seconds per shot for one kernel circuit.
    @param n_samples           Number of training samples.
    @param shots               Measurement shots per circuit.
    @return                    circuit_duration_s * shots * n_pairs.
    @note  n_pairs = n*(n-1)//2 (upper triangle; diagonal = 1.0 by definition).
    """
    n_pairs = n_samples * (n_samples - 1) // 2
    return circuit_duration_s * shots * n_pairs
# end def _qpu_seconds


# â”€â”€ Section 1: Current configuration estimate â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("=" * 70)
print(f"CURRENT CONFIG:  feature_map={FEATURE_MAP}  layers={LAYERS}"
      f"  PCA_COMPONENTS={PCA_COMPONENTS}  shots={DQC1_KERNEL_SHOTS}")
print("=" * 70)

_dur_s = _circuit_duration_seconds(PCA_COMPONENTS, FEATURE_MAP, LAYERS, _fake)
print(f"  Circuit duration: {_dur_s * 1e6:.2f} us/shot")
print()

# Table of QPU time vs N_SAMPLES for the current config
print(f"QPU time vs sample count  (feature_map={FEATURE_MAP}, "
      f"PCA={PCA_COMPONENTS} qubits, shots={DQC1_KERNEL_SHOTS}):")
print(f"  {'N':>6}  {'Pairs':>8}  {'Seconds':>10}  {'Minutes':>8}  Status")
print(f"  {'------':>6}  {'--------':>8}  {'----------':>10}  {'--------':>8}  ------")
for _n in [10, 20, 50, 100, 200, 500]:
    _secs = _qpu_seconds(_dur_s, _n)
    _pairs = _n * (_n - 1) // 2
    _status = "EXCEEDS LIMIT" if _secs > IBM_QPU_LIMIT_SECONDS else "within limit"
    print(f"  {_n:>6}  {_pairs:>8}  {_secs:>10.1f}  {_secs/60:>8.2f}  {_status}")
print()

# Specific result for the configured N_SAMPLES
if N_SAMPLES > 0:
    _total_s = _qpu_seconds(_dur_s, N_SAMPLES)
    print(f"Calculated IBM Compute time: {_total_s:.1f} seconds  "
          f"({_total_s / 60:.2f} min)"
          f"  [N={N_SAMPLES}, shots={DQC1_KERNEL_SHOTS}]")
    if _total_s > IBM_QPU_LIMIT_SECONDS:
        print(f"  WARNING: Exceeds IBM open plan limit ({IBM_QPU_LIMIT_SECONDS}s / 10 min).")
        print(f"  Reduce N_SAMPLES or DQC1_KERNEL_SHOTS before deploying.")
    else:
        print(f"  OK: Within IBM open plan limit ({IBM_QPU_LIMIT_SECONDS}s).")
else:
    print("Calculated IBM Compute time: set N_SAMPLES > 0 for a specific estimate.")
    print(f"  (N_SAMPLES=0 uses the full dataset â€” size unknown until Cell 9 loads data.)")
print()


# â”€â”€ Section 2: PCA_COMPONENTS comparison â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# Raw input vector size (256-dim, 1024-dim, etc.) does NOT affect QPU time â€”
# PCA always reduces inputs to PCA_COMPONENTS before the circuit is built.
# What matters is PCA_COMPONENTS (= qubit count).  The ZZ feature map uses
# O(n^2) RZZ gates per layer, so circuit depth grows quadratically with qubits.
print("=" * 70)
print("PCA_COMPONENTS COMPARISON  (how qubit count drives QPU time)")
print(f"  Raw input size is irrelevant â€” PCA reduces ALL input sizes to")
print(f"  PCA_COMPONENTS dims before the circuit.  Changing 256â†’1024 raw")
print(f"  features has zero effect on QPU time unless PCA_COMPONENTS changes.")
print("=" * 70)

# Fixed reference point: N=50 samples, current shots, current layers/feature_map
_REF_N = 50
print(f"Reference: N={_REF_N} samples, shots={DQC1_KERNEL_SHOTS}, "
      f"feature_map={FEATURE_MAP}, layers={LAYERS}")
print()
print(f"  {'PCA_COMP':>8}  {'Qubits':>6}  {'us/shot':>8}  "
      f"{'Secs (N={_REF_N})':>14}  {'vs limit':>12}  Status")
print(f"  {'--------':>8}  {'------':>6}  {'--------':>8}  "
      f"{'---------------':>14}  {'------------':>12}  ------")

for _pca in [4, 6, 8, 10, 12, 16]:
    _d = _circuit_duration_seconds(_pca, FEATURE_MAP, LAYERS, _fake)
    _secs = _qpu_seconds(_d, _REF_N)
    _pct = (_secs / IBM_QPU_LIMIT_SECONDS) * 100
    _marker = " <-- current" if _pca == PCA_COMPONENTS else ""
    _status = "EXCEEDS" if _secs > IBM_QPU_LIMIT_SECONDS else "OK"
    print(f"  {_pca:>8}  {_pca:>6}  {_d*1e6:>8.2f}  "
          f"  {_secs:>12.1f}    {_pct:>9.1f}%  {_status}{_marker}")


In [ ]:
# ============================================================
# LOCAL EXECUTION RESUMES
# ============================================================
# All cells below run locally regardless of USE_IBM_BACKEND.
# The kernel matrix K returned by build_dqc1_kernel_matrix()
# is identical in shape (N, N) whether computed locally or on IBM.
# Clustering, evaluation, and plotting are unaffected by the
# execution path chosen above.
print("[LOCAL] Resuming local execution after quantum kernel section.")


In [ ]:
# Override DQC1 purity circuit wiring to match gate arity
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister


def _infer_work_and_ancillas(U_A, ancilla_qubits=None):
    """@brief Infer the number of work qubits and ancilla qubits for DQC1 purity circuit construction.

    @param U_A            Quantum circuit or gate representing the feature map unitary.
    @param ancilla_qubits Optional number of ancilla qubits; defaults to 1 if not specified.
    @return               Tuple (gate_qubits, n_anc) where gate_qubits is the number of qubits in U_A
                          and n_anc is the number of ancilla qubits.
    @note                 Ensures at least 1 ancilla qubit is used for controlled operations.
    """
    gate_qubits = U_A.num_qubits if hasattr(U_A, "num_qubits") else U_A.to_gate().num_qubits
    n_anc = max(1, int(ancilla_qubits)) if ancilla_qubits is not None else 1
    return gate_qubits, n_anc
# end def _infer_work_and_ancillas

def construct_dqc1_circuit(U_A, rho_dim: int, ancilla_qubits=None):
    """@brief Construct the DQC1 purity measurement circuit with ancilla and work qubits.

    @param U_A            Quantum circuit or gate representing the feature map unitary.
    @param rho_dim         Dimensionality of the density matrix (unused in this implementation).
    @param ancilla_qubits Optional number of ancilla qubits; defaults to 1 if not specified.
    @return               Tuple (qc, qr_anc, qr_work) where qc is the constructed circuit,
                          qr_anc is the ancilla quantum register, and qr_work is the work register.
    @note                 Applies Hadamard gates to ancillas, controlled unitary, and final Hadamards.
    """
    n_work, n_anc = _infer_work_and_ancillas(U_A, ancilla_qubits=ancilla_qubits)
    qr_anc = QuantumRegister(n_anc, "anc")
    qr_work = QuantumRegister(n_work, "work")
    qc = QuantumCircuit(qr_anc, qr_work)
    gate_UA = _as_controllable_gate(U_A)

    for q in qr_anc:
        qc.h(q)
    # end for q   

    ctrl_qubits = list(qr_anc)
    work_qubits = list(qr_work)
    qc.append(gate_UA.control(len(ctrl_qubits)), ctrl_qubits + work_qubits)

    for q in qr_anc:
        qc.h(q)
    # end for q in qr_anc    
    
    return qc, qr_anc, qr_work
# end def construct_dqc1_circuit

def measure_dqc1_purity_sv(U_A, rho_dim: int, ancilla_qubits=None) -> float:
    """@brief Measure DQC1 purity using exact statevector simulation.

    @param U_A            Quantum circuit or gate representing the feature map unitary.
    @param rho_dim         Dimensionality of the density matrix (unused in this implementation).
    @param ancilla_qubits Optional number of ancilla qubits; defaults to 1 if not specified.
    @return               Purity value as the probability of measuring all ancilla qubits in |0>.
    @note                 Uses statevector simulation for exact computation without shots.
    """
    qc, qr_anc, qr_work = construct_dqc1_circuit(U_A, rho_dim=rho_dim, ancilla_qubits=ancilla_qubits)
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities()
    p_zero_anc = 0.0
    n_anc = len(qr_anc)
    anc_indices = [qc.qubits.index(q) for q in qr_anc]
    for outcome, p in enumerate(probs):
        if p < 1e-14:
            continue
        bits = format(outcome, f"0{qc.num_qubits}b")
        anc_bits = "".join(bits[i] for i in anc_indices)
        if anc_bits == "0" * n_anc:
            p_zero_anc += p
    return float(p_zero_anc)
# end def measure_dqc1_purity_sv

def _run_counts_in_chunks(backend, tqc, total_shots: int, chunk: int = 128) -> dict:
    """@brief Run a circuit in shot batches to stay within simulator memory limits.

    @param backend      AerSimulator backend to execute against.
    @param tqc          Transpiled quantum circuit to run.
    @param total_shots  Total number of measurement shots required.
    @param chunk        Maximum shots per individual backend.run() call.
    @return             Dict mapping bitstring outcomes to aggregated counts.
    @note   Uses the legacy backend.run() API valid for AerSimulator only.
            Only reached when DISABLE_REDUCTION=0 and shot-based purity is requested.
    """
    remaining = int(total_shots)
    agg = Counter()
    # Issue shots in chunks and accumulate counts across batches
    while remaining > 0:
        s = min(chunk, remaining)
        res = backend.run(tqc, shots=s).result()
        cnts = res.get_counts()
        # backend.run() may return a list of dicts for batched jobs
        cnts = cnts[0] if isinstance(cnts, list) else cnts
        agg.update(cnts)
        remaining -= s
    return dict(agg)
# end def _run_counts_in_chunks


def measure_dqc1_purity_shots(U_A, rho_dim: int, shots: int, ancilla_qubits=None) -> float:
    """@brief Measure DQC1 purity using shot-based simulation on a quantum simulator.

    @param U_A            Quantum circuit or gate representing the feature map unitary.
    @param rho_dim         Dimensionality of the density matrix (unused in this implementation).
    @param shots           Number of measurement shots to perform.
    @param ancilla_qubits Optional number of ancilla qubits; defaults to 1 if not specified.
    @return               Purity value as the fraction of shots where all ancilla qubits are measured in |0>.
    @note                 Uses AerSimulator with shot-based execution for approximate computation.
    """
    qc, qr_anc, qr_work = construct_dqc1_circuit(U_A, rho_dim=rho_dim, ancilla_qubits=ancilla_qubits)
    cr_anc = ClassicalRegister(len(qr_anc), "c_anc")
    qc.add_register(cr_anc)
    qc.measure(qr_anc, cr_anc)
    if not _AER_AVAILABLE:
        raise ImportError("qiskit-aer is required for shot-based purity but is not available on this platform (ARM64). Use statevector purity (DQC1_EXACT_KERNEL=1) or run on x64.")
    backend = AerSimulator(method="statevector")
    tqc = transpile(qc, backend=backend)
    counts = _run_counts_in_chunks(backend, tqc, total_shots=shots, chunk=128)
    zero_str = "0" * len(qr_anc)
    zeros = 0
    for outcome, cnt in counts.items():
        if outcome.startswith(zero_str):
            zeros += cnt
    return float(zeros) / shots
# end def measure_dqc1_purity_shots

In [ ]:
# With DISABLE_REDUCTION=1 (default), the no-op override at the bottom applies
# and the fast reduction functions above are never called.

# Cell: Override reduction to use fast kernel-purity entropy (no DQC1 in reduction)
import types

def _reduce_fast(self, X_arr, clusters):
    """@brief Fast reduction method relying on kernel purity to score moves and merges.

    @param self      The instance of the class.
    @param X_arr     The input array.
    @param clusters  The list of clusters.
    @return          The reduced clusters.
    @note            This method performs iterative reduction until no more merges are possible.
    """
    # Initialize clusters as lists, filtering out empty ones to start clean
    clusters = [list(c) for c in clusters if c]
    # Record initial total entropy for logging reduction progress
    entropy_log = [self._S2_total_fast(self.M_full, clusters)]
    # Iteratively reduce clusters until only one remains or no merges possible
    while len(clusters) > 1:
        # Flag to track if any move or merge occurred in this iteration
        merged = False
        # Attempt to move each data point to a better cluster if uniquely assigned
        for idx in range(len(X_arr)):
            # Find clusters containing this point (should be exactly one for valid move)
            memberships = [k for k, c in enumerate(clusters) if idx in c]
            if len(memberships) != 1:
                continue
            # Current cluster index for this point
            k_now = memberships[0]
            # List of other clusters to consider for move
            remaining = [c for k, c in enumerate(clusters) if k != k_now]
            # Find the best cluster to move to by minimizing entropy change
            best_k, best_dH = None, float("inf")
            for k, c in enumerate(remaining):
                dH = self._delta_H_fast(self.M_full, c, idx)
                if dH < best_dH:
                    best_k, best_dH = k, dH
            # Perform the move if a better cluster was found
            if best_k is not None:
                if len(clusters[k_now]) > 1:
                    clusters[k_now].remove(idx)
                else:
                    clusters[k_now] = []
                remaining[best_k].append(idx)
                clusters = remaining
                merged = True
                break
        # If no moves were possible, attempt to merge clusters
        if not merged:
            can_merge = False
            # For each cluster, find the best merge partner by minimizing entropy increase
            for i, ci in enumerate(clusters):
                if not ci:
                    continue
                best_j, best_dH = None, float("inf")
                for j, cj in enumerate(clusters):
                    if j == i or not cj:
                        continue
                    test_merge = ci + cj
                    H_sep = self._H2_cluster_fast(self.M_full, ci) + self._H2_cluster_fast(self.M_full, cj)
                    H_merged = self._H2_cluster_fast(self.M_full, test_merge)
                    dH_merge = H_merged - H_sep
                    if dH_merge < best_dH:
                        best_dH = dH_merge
                        best_j = j
                # Perform the merge if a suitable partner was found
                if best_j is not None:
                    clusters[i] = clusters[i] + clusters[best_j]
                    clusters[best_j] = []
                    clusters = [c for c in clusters if c]
                    can_merge = True
                    break
            # If no merges possible, stop reduction
            if not can_merge:
                break
        # Filter out empty clusters after moves/merges
        clusters = [c for c in clusters if c]
        # Log entropy after this reduction step
        entropy_log.append(self._S2_total_fast(self.M_full, clusters))
    # Store the entropy log for analysis
    self.entropy_log_ = entropy_log
    return clusters
# end def _reduce_fast

def _reduce_one_step_fast(self, clusters):
    """@brief Perform one step of fast reduction by attempting moves and merges to minimize entropy.

    @param self      The instance of the class.
    @param clusters  The list of current clusters.
    @return          The updated list of clusters after one reduction step.
    @note            This method tries to move points or merge clusters if beneficial, returning early on success.
    """
    # Create a copy of clusters as lists, filtering out any empty ones to ensure clean starting state
    clusters = [list(c) for c in clusters if c]
    # If only one or fewer clusters remain, no reduction is possible, so return as is
    if len(clusters) <= 1:
        return clusters
    # Attempt to move each data point to a better cluster to reduce entropy
    for idx in range(len(self.X_)):
        # Identify clusters containing this point; only proceed if exactly one (valid for move)
        memberships = [k for k, c in enumerate(clusters) if idx in c]
        if len(memberships) != 1:
            continue
        # Get the current cluster index for this point
        k_now = memberships[0]
        # List other clusters as potential move targets
        remaining = [c for k, c in enumerate(clusters) if k != k_now]
        # Initialize tracking for the best move: cluster index and entropy change
        best_k, best_dH = None, float("inf")
        # Evaluate entropy change for moving to each other cluster
        for k, c in enumerate(remaining):
            dH = self._delta_H_fast(self.M_full, c, idx)
            if dH < best_dH:
                best_k, best_dH = k, dH
        # If a better cluster was found, perform the move and return updated clusters
        if best_k is not None:
            # Remove point from current cluster, avoiding empty list if it's the last point
            if len(clusters[k_now]) > 1:
                clusters[k_now].remove(idx)
            else:
                clusters[k_now] = []
            # Add point to the best target cluster
            remaining[best_k].append(idx)
            # Update clusters list and filter empties
            clusters = remaining
            return [c for c in clusters if c]
    # If no moves succeeded, attempt to merge clusters to reduce entropy
    for i, ci in enumerate(clusters):
        # Skip empty clusters
        if not ci:
            continue
        # Initialize tracking for the best merge: partner index and entropy change
        best_j, best_dH = None, float("inf")
        # Evaluate entropy change for merging with each other cluster
        for j, cj in enumerate(clusters):
            if j == i or not cj:
                continue
            # Simulate the merge and compute entropy difference
            test_merge = ci + cj
            H_sep = self._H2_cluster_fast(self.M_full, ci) + self._H2_cluster_fast(self.M_full, cj)
            H_merged = self._H2_cluster_fast(self.M_full, test_merge)
            dH_merge = H_merged - H_sep
            if dH_merge < best_dH:
                best_dH = dH_merge
                best_j = j
        # If a beneficial merge was found, perform it and return updated clusters
        if best_j is not None:
            # Merge the clusters and clear the partner
            clusters[i] = clusters[i] + clusters[best_j]
            clusters[best_j] = []
            # Filter out empties and return
            return [c for c in clusters if c]
    # If no moves or merges were possible, return clusters unchanged
    return [c for c in clusters if c]
# end def _reduce_one_step_fast

# fast reduction
DQC1EntropyCluster._reduce = _reduce_fast
DQC1EntropyCluster._reduce_one_step = _reduce_one_step_fast

# OVERRIDE: Disable reduction entirely to preserve K from init phase
# This prevents cluster collapse from K=INIT_CLUSTERS down to K=2
def _reduce_disabled(self, X_arr, clusters):
    """No-op reduction: return initial clusters unchanged.
    This preserves K from the initialization phase and prevents entropy-driven collapse."""
    clusters = [list(c) for c in clusters if c]
    # Just record entropy log but don't merge any clusters
    self.entropy_log_ = [sum(len(c) for c in clusters)]  # Single entry: total items in clusters
    return clusters
# end def _reduce_disabled

def _reduce_one_step_disabled(self, clusters):
    """No-op one-step reduction: return clusters unchanged."""
    return [list(c) for c in clusters if c]
# end def _reduce_one_step_disabled

# Apply disabled reduction override (set to True to enable init-only clustering)
if DISABLE_REDUCTION:
    DQC1EntropyCluster._reduce = _reduce_disabled
    DQC1EntropyCluster._reduce_one_step = _reduce_one_step_disabled
    print("[CONFIG] Reduction phase DISABLED - using init labels only to preserve K")
# end if DISABLE_REDUCTION

In [ ]:
# =============================================================================
# @file     DQC1.ipynb -- MRI data load and evaluation cell
# @brief    Loads MRI features/labels, runs DQC1 clustering,
#           and reports clustering performance metrics against ground truth.
#
# @license  MIT License -- see LICENSE file for details.
# =============================================================================
from sklearn.preprocessing import StandardScaler

# ---- MRI DATA PATHS ----
# NOTE: This notebook is MRI-only; UCI loaders are disabled.
# Resolve paths: check notebook directory first, then data/ subdirectory, then env vars

def _resolve_data_path(filename: str, default_rel_path: str = None) -> str:
    """@brief Resolve a data file path by probing several candidate locations.

    @param filename         Bare filename to check in the current working directory.
    @param default_rel_path Relative path (e.g. data/<dataset>/...) to try next.
    @return                 First path that exists on disk, or bare filename as fallback.
    """
    # Try direct filename in current working directory
    if os.path.exists(filename):
        return filename

    if default_rel_path and os.path.exists(default_rel_path):
        return default_rel_path

    # Return the one that's most likely to work (current dir first)
    return filename
# end def _resolve_data_path


def _get_mri_dataset_folders(data_root: str = "data") -> list[str]:
    """@brief Discover available MRI dataset folders under the top-level data directory."""
    root = os.path.abspath(data_root)
    if not os.path.isdir(root):
        return []
    return sorted(
        name for name in os.listdir(root)
        if os.path.isdir(os.path.join(root, name))
    )


def _prompt_select_dataset(datasets: list[str]) -> str:
    """@brief Prompt the user to select one dataset folder from the discovered list."""
    if not datasets:
        raise FileNotFoundError(f"No dataset folders found under {os.path.abspath('data')}")
    if len(datasets) == 1:
        print(f"Using only available dataset folder: {datasets[0]}")
        return datasets[0]

    env_dataset = os.environ.get("MRI_DATASET")
    if env_dataset and env_dataset in datasets:
        print(f"Using dataset from MRI_DATASET={env_dataset}")
        return env_dataset

    print("Available MRI dataset folders:")
    for idx, name in enumerate(datasets, start=1):
        print(f"  {idx}. {name}")

    while True:
        selection = input(
            f"Select dataset folder by number (1-{len(datasets)}) or name: "
        ).strip()
        if selection.isdigit():
            idx = int(selection) - 1
            if 0 <= idx < len(datasets):
                return datasets[idx]
        if selection in datasets:
            return selection
        print("Invalid selection. Please choose a valid number or folder name.")


DATASET_FOLDER = _prompt_select_dataset(_get_mri_dataset_folders("data"))
MRI_DATASET_NAME = DATASET_FOLDER

MRI_TRAIN_FEATURES = os.environ.get(
    "MRI_TRAIN_FEATURES",
    _resolve_data_path(
        f"{MRI_DATASET_NAME}_train.bin",
        os.path.join("data", DATASET_FOLDER, "features", f"{MRI_DATASET_NAME}_train.bin"),
    ),
)
MRI_TRAIN_LABELS = os.environ.get(
    "MRI_TRAIN_LABELS",
    _resolve_data_path(
        f"{MRI_DATASET_NAME}_train.meta",
        os.path.join("data", DATASET_FOLDER, "labels", f"{MRI_DATASET_NAME}_train.meta"),
    ),
)
MRI_TRAIN_DATASET = os.environ.get("MRI_TRAIN_DATASET", f"{MRI_DATASET_NAME}_train")

# ---- HELPERS ----

def load_meta_labels(path: str) -> np.ndarray:
    """@brief Load integer class labels from a plain-text .meta file.

    @param path  Path to the label file (one integer per line).
    @return      1-D int ndarray of class labels.
    """
    arr = np.loadtxt(path, dtype=int)
    return arr
# end def load_meta_labels

def load_features(path: str) -> np.ndarray:
    """@brief Load feature vectors from a .npy or .bin file.

    @param path  Path to the feature file.
    @return      (n_samples, n_features) float64 ndarray.
    @note  .bin files carry a 32-float32 header that is skipped.
           Assumed feature dimension is 256 (C-order / row-major layout).
    """
    if path.endswith('.npy'):
        # NumPy native format -- straightforward load
        X = np.load(path, allow_pickle=False)
        return np.asarray(X, float)
    elif path.endswith('.bin'):
        # Binary float32 file; first 32 values are a header -- skip them
        X_flat = np.fromfile(path, dtype=np.float32)
        n_features = 256
        X_data = X_flat[32:]  # Skip 32-value header
        n_samples = len(X_data) // n_features
        if len(X_data) % n_features != 0:
            raise ValueError(f"Data size {len(X_data)} is not divisible by feature dimension {n_features}")
        X = X_data[:n_samples * n_features].reshape((n_samples, n_features))
        return np.asarray(X, float)
    else:
        # Unknown extension: try npy first, then binary with header skip
        try:
            X = np.load(path, allow_pickle=False)
            return np.asarray(X, float)
        except Exception:
            X_flat = np.fromfile(path, dtype=np.float32)
            # Attempt with 32-value header skip
            try:
                n_features = 256
                X_data = X_flat[32:]
                n_samples = len(X_data) // n_features
                if len(X_data) % n_features == 0:
                    X = X_data.reshape((n_samples, n_features))
                    return np.asarray(X, float)
            except Exception:
                pass
            # Final fallback: no header skip
            n_features = 256
            n_samples = len(X_flat) // n_features
            if len(X_flat) % n_features != 0:
                raise ValueError(f"File size {len(X_flat)} is not divisible by feature dimension {n_features}")
            X = X_flat.reshape((n_samples, n_features))
            return np.asarray(X, float)
# end def load_features

def maybe_pca(X: np.ndarray, n_comp: int, pca_fitted=None):
    """@brief Optionally apply PCA dimensionality reduction.

    @param X          Input feature matrix.
    @param n_comp     Number of principal components to keep.
    @param pca_fitted Pre-fitted PCA object to reuse; if None, a new PCA is fitted on X.
    @return           Tuple (X_reduced, suffix_string, pca_obj).
                      pca_obj is the fitted PCA instance for reuse on other splits.
                      suffix_string is empty when PCA is off.
    @note  Always pass pca_fitted=<train_pca> when transforming the test set so both
           splits share the same projection axes and distance comparisons are valid.
    """
    if not USE_PCA:
        return X, "", None
    if pca_fitted is None:
        # Fit a new PCA on the provided data; always use this path for training data
        pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
        Xr = pca.fit_transform(X)
    else:
        # Reuse the pre-fitted PCA axes â€” avoids leaking test distribution into the projection
        pca = pca_fitted
        Xr = pca.transform(X)
    return Xr, f"_pca{n_comp}", pca
# end def maybe_pca

def fit_scaler(X_train: np.ndarray) -> StandardScaler:
    """@brief Fit a StandardScaler on training data.

    @param X_train  Training feature matrix used to compute mean/std.
    @return         Fitted StandardScaler instance.
    """
    sc = StandardScaler()
    sc.fit(X_train)
    return sc
# end def fit_scaler

def scale_dataset(sc: StandardScaler, X: np.ndarray) -> np.ndarray:
    """@brief Apply a pre-fitted scaler to a feature matrix.

    @param sc  Fitted StandardScaler.
    @param X   Feature matrix to transform.
    @return    Scaled feature matrix.
    """
    return sc.transform(X)
# end def scale_dataset

def maybe_noise(X: np.ndarray, sigma: float) -> np.ndarray:
    """@brief Optionally inject Gaussian noise into a feature matrix.

    @param X      Feature matrix to perturb.
    @param sigma  Standard deviation of noise; pass 0 to skip.
    @return       Perturbed (or unchanged) feature matrix.
    """
    if sigma <= 0:
        return X
    noise = np.random.normal(0, sigma, size=X.shape)
    return X + noise
# end def maybe_noise

# ---- Metrics ----

def _bcubed_precision_recall(y_true: np.ndarray, y_pred: np.ndarray):
    """@brief Compute B-Cubed precision, recall, and F-score.

    @param y_true  Ground-truth class labels.
    @param y_pred  Predicted cluster assignments.
    @return        Tuple (precision, recall, f1_score).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    n = len(y_true)
    if n == 0 or n != len(y_pred):
        return 0.0, 0.0, 0.0
    pair_counts    = Counter(zip(y_true, y_pred))
    cluster_counts = Counter(y_pred)
    class_counts   = Counter(y_true)
    prec_sum = 0.0
    rec_sum  = 0.0
    for (lbl, clu), cnt in pair_counts.items():
        # Each item contributes its within-cluster/class overlap fraction
        prec_sum += cnt * (cnt / cluster_counts[clu])
        rec_sum  += cnt * (cnt / class_counts[lbl])
    precision = prec_sum / n
    recall    = rec_sum  / n
    fscore    = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return precision, recall, fscore
# end def _bcubed_precision_recall


def eval_scores(X: np.ndarray, y_true: np.ndarray, y_pred: np.ndarray):
    """@brief Compute NMI, Pairwise F-score (Fowlkes-Mallows), and B-Cubed F-score.

    @param X       Feature matrix (unused; kept for call-site compatibility).
    @param y_true  Ground-truth class labels.
    @param y_pred  Predicted cluster assignments.
    @return        Dict with keys: k, nmi, fm, b3_f.
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    K    = len(np.unique(y_pred))
    # Normalized Mutual Information â€” shared information between clusters and classes
    nmi  = normalized_mutual_info_score(y_true, y_pred)
    # Fowlkes-Mallows â€” geometric mean of pairwise precision and recall
    fm   = fowlkes_mallows_score(y_true, y_pred)
    # B-Cubed F-score â€” per-item precision/recall aggregated to F-score
    _, _, b3_f = _bcubed_precision_recall(y_true, y_pred)
    return {
        "k":    int(K),
        "nmi":  float(nmi),
        "fm":   float(fm),
        "b3_f": float(b3_f),
    }
# end def eval_scores

def print_scores(label: str, scores: dict):
    """@brief Format and emit the three clustering metrics via dlog.

    @param label   Short descriptor shown at the start of the line.
    @param scores  Dict returned by eval_scores().
    @note  Output goes to both the console and the session log file via dlog().
    """
    def _fmt(x, decimals=3):
        if x is None or (isinstance(x, float) and __import__("math").isnan(x)):
            return "nan"
        return f"{x:.{decimals}f}"
    # Emit through dlog so the line appears in both console and log file
    dlog(
        f"{label:<35} | K={scores['k']:2d} | "
        f"NMI={_fmt(scores['nmi'])} "
        f"PW-F={_fmt(scores['fm'])} "
        f"B3-F={_fmt(scores['b3_f'])}"
    )
# end def print_scores

def _elapsed(t0: datetime.datetime) -> str:
    """@brief Return a human-readable elapsed-time string since t0.

    @param t0  Start timestamp from datetime.datetime.now().
    @return    String of the form "Xs" or "Xm Ys".
    """
    secs = (datetime.datetime.now() - t0).total_seconds()
    m, s = divmod(int(secs), 60)
    return f"{m}m {s}s" if m else f"{s}s"
# end def _elapsed

def evaluate_mri():
    """@brief Full MRI evaluation pipeline: load data, cluster, report metrics.

    @return  Tuple (X_tr_p, y_tr, y_pred) -- preprocessed train features,
             ground-truth labels, and DQC1 cluster assignments.
    @note    Runs DQC1 quantum clustering and reports performance metrics
             against ground-truth labels. All output is emitted via dlog().
    """
    # Mark the start of the full pipeline for overall elapsed-time tracking
    _t_total = datetime.datetime.now()

    dlog("[MRI] ============================================================")
    dlog("[MRI] DQC1 quantum clustering pipeline")
    dlog("[MRI] ============================================================")
    dlog(f"[MRI] START         : {_t_total.isoformat()}")
    dlog(f"[MRI] CONFIG: FEATURE_MAP={FEATURE_MAP}, LAYERS={LAYERS}, INIT_CLUSTERS={INIT_CLUSTERS}, "
         f"COMP_RANK={DQC1_COMP_RANK}, DISABLE_REDUCTION={DISABLE_REDUCTION}")

    # -- Data loading ---------------------------------------------------------
    dlog("[MRI] ---- STEP 1/3: Loading data ----")
    dlog(f"[MRI] Data path resolution:")
    dlog(f"  Train features: {MRI_TRAIN_FEATURES}")
    dlog(f"  Train labels:   {MRI_TRAIN_LABELS}")
    dlog(f"[MRI] Current working directory: {os.getcwd()}")
    dlog("[MRI] Checking file existence...")
    for label, path in [("train features", MRI_TRAIN_FEATURES),
                        ("train labels",   MRI_TRAIN_LABELS)]:
        exists = os.path.exists(path)
        dlog(f"  {label}: {'EXISTS' if exists else 'MISSING'} {path}")

    _t0 = datetime.datetime.now()
    X_tr = load_features(MRI_TRAIN_FEATURES)
    y_tr = load_meta_labels(MRI_TRAIN_LABELS)
    dlog(f"[MRI] Data loaded in {_elapsed(_t0)} -- train {X_tr.shape}")

    # Optionally subsample the training set to control IBM QPU cost or speed up local tests
    if N_SAMPLES > 0 and N_SAMPLES < len(X_tr):
        _orig_n = len(X_tr)
        # Use a fixed seed so the same subset is chosen on every run
        _sub_rng = np.random.RandomState(RANDOM_STATE)
        _sub_idx = _sub_rng.choice(_orig_n, N_SAMPLES, replace=False)
        X_tr = X_tr[_sub_idx]
        y_tr = y_tr[_sub_idx]
        dlog(f"[MRI] N_SAMPLES={N_SAMPLES}: subsampled train to {N_SAMPLES} of {_orig_n} samples")

    # -- Preprocessing --------------------------------------------------------
    dlog("[MRI] ---- STEP 2/3: Preprocessing (noise / scale / PCA) ----")
    _t0 = datetime.datetime.now()
    if DATA_NOISE > 0:
        # Inject Gaussian noise to test robustness
        X_tr = maybe_noise(X_tr, DATA_NOISE)
        dlog(f"[MRI] Noise injected (sigma={DATA_NOISE})")

    # Fit scaler on training data only
    sc     = fit_scaler(X_tr)
    X_tr_s = scale_dataset(sc, X_tr)

    # Fit PCA on training data
    X_tr_p, pca_suffix, _ = maybe_pca(X_tr_s, PCA_COMPONENTS)
    dlog(f"[MRI] Preprocessing done in {_elapsed(_t0)} -- "
         f"shape: {X_tr_p.shape}; PCA suffix={pca_suffix!r}")

    # -- DQC1 quantum clustering ----------------------------------------------
    dlog("[MRI] ---- STEP 3/3: DQC1 quantum clustering (may take several minutes) ----")
    dqc = DQC1EntropyCluster(
        init_clusters=INIT_CLUSTERS,
        n_init_per_seed=N_INIT,
        kernel_shots=DQC1_KERNEL_SHOTS,
        exact_sv_kernel=bool(DQC1_EXACT_KERNEL),
        random_state=RANDOM_STATE,
        compress_rank=DQC1_COMP_RANK if DQC1_COMP_RANK > 0 else None,
    )
    dlog(f"[MRI] DQC1 fit_full() starting at {datetime.datetime.now().isoformat()} ...")
    _t0 = datetime.datetime.now()
    dqc.fit_full(X_tr_p)
    dlog(f"[MRI] DQC1 fit_full() completed in {_elapsed(_t0)}")

    # Use init labels -- reduction is disabled to preserve K from initialisation
    y_pred = dqc.labels_full_
    sc_dqc = eval_scores(X_tr_p, y_tr, y_pred)
    print_scores("[MRI][DQC1]", sc_dqc)

    dlog(f"[MRI] ---- Pipeline complete -- total elapsed: {_elapsed(_t_total)} ----")
    dlog("[MRI] ============================================================")

    # Return preprocessed features, ground-truth labels, and DQC1 cluster assignments
    return X_tr_p, y_tr, y_pred
# end def evaluate_mri


In [ ]:
# Run MRI pipeline
X_tr_p, y_tr, y_pred = evaluate_mri()

# Ground truth and cluster assignment summary
print("Ground truth analysis:")
print(f"  Unique classes: {len(np.unique(y_tr))}")
print(f"  Class distribution: {Counter(y_tr)}")
print(f"  Train samples: {len(y_tr)}")
print("DQC1 cluster assignment:")
print(f"  Clusters found: {len(np.unique(y_pred))}")
print(f"  Cluster distribution: {Counter(y_pred)}")
